# MAna.data - Reading, Cleaning, and Preparing Real Data

This documentation uses a real sample from old work in `Womens Clothing E-Commerce Reviews.csv`. The full dataset is large enough to keep outside the docs, so the repository carries a small chunk.

If you are editing MAna locally, install it in editable mode first so the notebook imports the source code you are changing:

```bash
python -m pip install -e .
```

## What this module covers

`MAna.data` has two layers:

- `data_io.py`: flexible readers for tables, clipboard data, image folders, SQL queries, and HTML tables.
- `data_cleaning.py`: focused cleaning functions plus `DataCleaner`, a chainable cleaning pipeline with reports.

Use the small functions when one operation is clear. Use `DataCleaner` when the cleaning itself is part of the experiment and you want to inspect what happened.

In [2]:
from pathlib import Path
import sqlite3
import tempfile

import numpy as np
import pandas as pd

from MAna.data import (
    DataCleaner,
    drop_missing_values,
    fill_missing_values,
    group_and_aggregate,
    quick_clean,
    quick_clean_text_entries,
    read_data,
    read_from_database,
    read_images,
    remove_outliers,
    smart_clean,
    solve_data_entry_errors,
    strict_validate,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

## 1. Start with `read_data`

`read_data()` is the general tabular reader. It routes by file extension and passes extra keyword arguments to the matching pandas reader where that makes sense.

Supported inputs include CSV, Excel, JSON, TXT, clipboard, SAS, SPSS, Parquet, and Pickle. Optional formats only import their optional dependencies when that branch is used, so a normal CSV read stays lightweight.

In [3]:
sample_path = "data/womens_clothing_reviews_sample.csv"
if not Path("data/womens_clothing_reviews_sample.csv").exists():
    sample_path = Path("docs/notebooks/data/womens_clothing_reviews_sample.csv")

reviews = read_data(sample_path)
reviews.shape

(160, 11)

In [4]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               160 non-null    int64 
 1   Clothing ID              160 non-null    int64 
 2   Age                      160 non-null    int64 
 3   Title                    112 non-null    object
 4   Review Text              139 non-null    object
 5   Rating                   160 non-null    int64 
 6   Recommended IND          160 non-null    int64 
 7   Positive Feedback Count  160 non-null    int64 
 8   Division Name            150 non-null    object
 9   Department Name          150 non-null    object
 10  Class Name               150 non-null    object
dtypes: int64(6), object(5)
memory usage: 13.9+ KB


In [5]:
reviews

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i ...",5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and really wanted it to work for me. i initially ...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfec...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...,...
155,761,886,44,NaN,Bought this today - cant speak to wear - but overall design is so incredibly cute and ...,5,1,21,General Petite,Tops,Knits
156,764,1087,34,Great for hot summers,"I am on the fence about this dress, as you'll see the reasons below, but it is really ...",4,1,42,General,Dresses,Dresses
157,774,1110,42,NaN,"Adorable, comfortable dress. the denim is super soft with a bit of stretch. please be ...",5,1,29,General Petite,Dresses,Dresses
158,824,886,32,Love the black,I bought this in black and it did require me to wear a shirt underneath as it was a li...,5,1,24,General Petite,Tops,Knits


In [6]:
pd.DataFrame(
    {
        "dtype": reviews.dtypes.astype(str),
        "missing": reviews.isna().sum(),
        "missing_percent": (reviews.isna().mean() * 100).round(1),
        "unique": reviews.nunique(dropna=True),
    }
)

,dtype,missing,missing_percent,unique
Unnamed: 0,int64,0,0.0,160
Clothing ID,int64,0,0.0,53
Age,int64,0,0.0,47
Title,object,48,30.0,111
Review Text,object,21,13.1,139
Rating,int64,0,0.0,5
Recommended IND,int64,0,0.0,2
Positive Feedback Count,int64,0,0.0,26
Division Name,object,10,6.2,3
Department Name,object,10,6.2,5


A quick read of the sample's info already gives us useful cleaning targets: missing review text, missing product hierarchy columns, human text, numeric feedback counts, and mixed naming conventions.

## 2. Show the IO flexibility

The same `read_data()` call can read other local table formats. The important idea is simple: keep the loading interface stable and let the file extension decide the reader.

In [7]:
small = reviews.head(8)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = tmpdir + "/reviews.csv"
    json_path = tmpdir + "/reviews.json"
    txt_path = tmpdir + "/reviews.txt"
    xlsx_path = tmpdir + "/reviews.xlsx"
    parquet_path = tmpdir + "/reviews.parquet"
    pickle_path = tmpdir + "/reviews.pickle"

    small.to_csv(csv_path, index=False)
    small.to_json(json_path, orient="records")
    small.to_csv(txt_path, sep="\t", index=False)
    small.to_excel(xlsx_path, index=False)
    small.to_parquet(parquet_path, index=False)
    small.to_pickle(pickle_path)

    loaded_shapes = {
        "csv": read_data(csv_path).shape,
        "json": read_data(json_path).shape,
        "txt": read_data(txt_path).shape,
        "xlsx": read_data(xlsx_path).shape,
        "parquet": read_data(parquet_path).shape,
        "pickle": read_data(pickle_path).shape,
    }

loaded_shapes

{'csv': (8, 11),
 'json': (8, 11),
 'txt': (8, 11),
 'xlsx': (8, 11),
 'parquet': (8, 11),
 'pickle': (8, 11)}

`read_from_database()` is for SQL-backed work. It returns a DataFrame directly from a query. Here we create a tiny SQLite database from the review sample and read it back through MAna.

In [8]:
with tempfile.TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / "reviews.sqlite"
    conn = sqlite3.connect(db_path)
    try:
        reviews[["Clothing ID", "Age", "Rating", "Recommended IND", "Department Name"]].to_sql(
            "reviews", conn, index=False, if_exists="replace"
        )
    finally:
        conn.close()

    db_url = "sqlite:///" + db_path.as_posix()
    high_rating_departments = read_from_database(
        db_url,
        """
        SELECT "Department Name" AS department,
               COUNT(*) AS reviews,
               AVG(Rating) AS avg_rating
        FROM reviews
        WHERE "Recommended IND" = 1
        GROUP BY "Department Name"
        ORDER BY reviews DESC
        """,
    )

high_rating_departments

,department,reviews,avg_rating
0,Tops,68,4.661765
1,Dresses,32,4.500000
2,Bottoms,10,4.400000
3,None,10,5.000000
4,Intimate,8,4.625000
5,Jackets,5,5.000000


`read_images()` loads a folder of `.jpg` and `.png` files into a single NumPy array. This is useful for quick classical ML experiments or sanity checks before a TensorFlow/PyTorch pipeline.

In [9]:
from PIL import Image

with tempfile.TemporaryDirectory() as tmpdir:
    image_dir = Path(tmpdir)
    Image.new("RGB", (12, 12), color=(220, 40, 40)).save(image_dir / "red.png")
    Image.new("RGB", (12, 12), color=(40, 90, 220)).save(image_dir / "blue.jpg")

    images = read_images(image_dir, image_size=(8, 8))

images.shape, images.dtype, images[0, 0, 0].tolist()

((2, 8, 8, 3), dtype('uint8'), [40, 91, 219])

`read_images_tf()` does the same job through TensorFlow's `image_dataset_from_directory`, then maps images to floats, caches, and prefetches. Keep it for real deep-learning image folders where TensorFlow is installed.

`read_from_html(url, table_index=0)` reads a table from a web page. It is intentionally not executed here because docs should run without internet access, but the usage is direct:

```python
tables = read_from_html("https://example.com/page-with-table", table_index=0)
```

## 3. Missing values: dropping vs filling

`drop_missing_values()` and `fill_missing_values()` are focused helpers. They are best when you already know which columns you are working on.

In [10]:
missing_demo = reviews[["Title", "Review Text", "Division Name", "Rating", "Positive Feedback Count"]]

rows_kept = drop_missing_values(missing_demo, threshold=0.8, axis=0) #removes rows with more than 20% missing values
cols_kept = drop_missing_values(missing_demo, threshold=0.9, axis=1) #removes columns with more than 10% missing values

{
    "original_shape": missing_demo.shape,
    "row_threshold_shape": rows_kept.shape,
    "column_threshold_shape": cols_kept.shape,
}

{'original_shape': (160, 5),
 'row_threshold_shape': (139, 5),
 'column_threshold_shape': (160, 3)}

In [11]:
numeric_filled = fill_missing_values(
    reviews[["Age", "Rating", "Positive Feedback Count"]],
    method="median",
) # replaces missing numeric values with the median of each column

text_filled = fill_missing_values(
    reviews[["Title", "Review Text"]],
    fill_value={"Title": "untitled", "Review Text": "no review text"},
) # replaces missing text values with specified default strings

numeric_filled.isna().sum().to_dict(), text_filled.isna().sum().to_dict()

({'Age': 0, 'Rating': 0, 'Positive Feedback Count': 0},
 {'Title': 0, 'Review Text': 0})

## 4. `solve_data_entry_errors()` in real scenarios

This function is for text columns. It can normalize case and spacing, remove special characters, fix common typos, map known wrong values, validate against allowed values, fuzzy-match close values, validate with a custom function, return a report, and optionally edit in place.

In [12]:
entry_demo = pd.DataFrame(
    {
        "department": [" Dresses ", "dresses", "Dreses", "Tops", "Trend", "Unknown Dept", None],
        "status": [" recieved ", "recieve", "RETURNED!!", "kept", "KEPT", "n/a", "kept"],
        "email": ["a@example.com", "wrong", "b@example.com", "missing-at.com", "c@example.com", "", None],
    }
)

entry_demo

,department,status,email
0,Dresses,recieved,a@example.com
1,dresses,recieve,wrong
2,Dreses,RETURNED!!,b@example.com
3,Tops,kept,missing-at.com
4,Trend,KEPT,c@example.com
5,Unknown Dept,n/a,
6,None,kept,None


In [13]:
# Scenario A: basic normalization only.
normalized, report = solve_data_entry_errors(
    entry_demo,
    column=["department", "status"],
    return_report=True,
) #solves data entry errors by normalizing text entries and providing a report of changes made

normalized, report["summary"]

(     department      status           email
 0       dresses    recieved   a@example.com
 1       dresses     receive           wrong
 2        dreses  returned!!   b@example.com
 3          tops        kept  missing-at.com
 4         trend        kept   c@example.com
 5  unknown dept         n/a                
 6          <NA>        kept            None,
 'Processed 2 columns, made 0 corrections')

In [14]:
# Scenario B: known mapping for business-specific variations.
status_mapping = {"recieved": "received", "receive": "received", "returned": "returned", "kept": "kept", "n/a": pd.NA}

mapped_status, mapping_report = solve_data_entry_errors(
    entry_demo,
    column="status",
    expected_values=status_mapping,
    remove_special_chars=True,
    return_report=True,
) #solves data entry errors by normalizing text entries, applying a known mapping for business-specific variations, and providing a report of changes made

mapped_status[["status"]], mapping_report

(     status
 0  received
 1  received
 2  returned
 3      kept
 4      kept
 5        na
 6      kept,
 {'columns_processed': ['status'],
  'total_fixes': 6,
  'errors_by_column': {'status': 6},
  'corrections_made': {'status': ['recieved -> received (1x)',
    'receive -> received (1x)',
    'returned -> returned (1x)',
    'kept -> kept (3x)']},
  'unfixable_values': {'status': []},
  'summary': 'Processed 1 columns, made 6 corrections'})

In [15]:
# Scenario C: valid-list enforcement with fuzzy matching.
department_clean, department_report = solve_data_entry_errors(
    entry_demo,
    column="department",
    expected_values=["dresses", "tops", "trend"],
    fuzzy_match=True,
    fuzzy_threshold=80,
    return_report=True,
) #solves data entry errors by normalizing text entries, enforcing a valid list of expected values with fuzzy matching for close matches, and providing a report of changes made

department_clean[["department"]], department_report

(  department
 0    dresses
 1    dresses
 2    dresses
 3       tops
 4      trend
 5       <NA>
 6       <NA>,
 {'columns_processed': ['department'],
  'total_fixes': 2,
  'errors_by_column': {'department': 2},
  'corrections_made': {'department': ['Fuzzy matched: dreses -> dresses (1x)']},
  'unfixable_values': {'department': ['unknown dept']},
  'summary': 'Processed 1 columns, made 2 corrections'})

In [16]:
# Scenario D: callable validation for rules that are easier as Python logic.
email_clean, email_report = solve_data_entry_errors(
    entry_demo,
    column="email",
    expected_values=lambda value: "@" in str(value) and "." in str(value),
    return_report=True,
) #solves data entry errors by normalizing text entries, enforcing a callable validation function for custom rules, and providing a report of changes made

email_clean[["email"]], email_report

(           email
 0  a@example.com
 1           <NA>
 2  b@example.com
 3           <NA>
 4  c@example.com
 5           <NA>
 6           <NA>,
 {'columns_processed': ['email'],
  'total_fixes': 4,
  'errors_by_column': {'email': 4},
  'corrections_made': {'email': ['Failed custom validation (4x)']},
  'unfixable_values': {'email': ['wrong', 'missing-at.com', '', <NA>]},
  'summary': 'Processed 1 columns, made 4 corrections'})

In [17]:
# Convenience wrappers for common habits.
quick = quick_clean_text_entries(entry_demo, "department", expected_values=["dresses", "tops", "trend"]) #solves data entry errors by normalizing text entries and enforcing a valid list of expected values
strict = strict_validate(entry_demo, "department", expected_values=["dresses", "tops", "trend"]) # solves data entry errors by normalizing text entries and enforcing a valid list of expected values, returning only valid entries
smart, smart_report = smart_clean(entry_demo, column=["department", "status"], return_report=True) #solves data entry errors by normalizing text entries, enforcing a valid list of expected values, and providing a report of changes made

{
    "quick_departments": quick["department"].tolist(),
    "strict_departments": strict["department"].tolist(),
    "smart_summary": smart_report["summary"],
}

{'quick_departments': ['dresses',
  'dresses',
  'dresses',
  'tops',
  'trend',
  <NA>,
  <NA>],
 'strict_departments': [<NA>, 'dresses', <NA>, <NA>, <NA>, <NA>, <NA>],
 'smart_summary': 'Processed 2 columns, made 0 corrections'}

## 5. Outlier removal methods

MAna exposes specific functions for each method and one dispatcher called `remove_outliers()`. The dispatcher is the easiest way to compare methods in experiments.

Available methods: `zscore`, `iqr`, `isolation_forest`, `lof`, `mad`, `dbscan`, and `percentile`.

In [18]:
outlier_frame = reviews[["Age", "Rating", "Positive Feedback Count"]].copy()
outlier_frame = pd.concat(
    [outlier_frame, pd.DataFrame({"Age": [120], "Rating": [1], "Positive Feedback Count": [250]})],
    ignore_index=True,
)

outlier_results = []
for method, kwargs in {
    "zscore": {"threshold": 3},
    "iqr": {"factor": 1.5},
    "mad": {"threshold": 3.5},
    "percentile": {"lower": 0.01, "upper": 0.99},
    "isolation_forest": {"contamination": 0.03, "random_state": 42},
    "lof": {"n_neighbors": 20, "contamination": 0.03},
    "dbscan": {"eps": 1.2, "min_samples": 5},
}.items():
    cleaned = remove_outliers(outlier_frame, method=method, **kwargs) #removes outliers from the dataset using various methods and parameters
    outlier_results.append({"method": method, "rows_kept": len(cleaned), "rows_removed": len(outlier_frame) - len(cleaned)})

pd.DataFrame(outlier_results).sort_values("rows_removed", ascending=False) #summarizes the results of outlier removal methods, showing how many rows were kept and removed for each method

,method,rows_kept,rows_removed
1,iqr,122,39
2,mad,132,29
4,isolation_forest,156,5
5,lof,156,5
3,percentile,157,4
6,dbscan,158,3
0,zscore,159,2


## 6. Grouping and pivoting

`group_and_aggregate()` is a tidy wrapper around a common pandas pattern.

In [19]:
department_summary = group_and_aggregate(
    reviews.dropna(subset=["Department Name"]),
    group_cols=["Department Name"],
    agg_dict={"Rating": ["mean", "count"], "Recommended IND": ["mean"], "Positive Feedback Count": ["mean"]},
) #groups the reviews by department and calculates summary statistics for each department, including average rating, count of ratings, average recommendation indicator, and average positive feedback count

department_summary.sort_values("Rating_count", ascending=False).head()

,Department Name,Rating_mean,Rating_count,Recommended IND_mean,Positive Feedback Count_mean
4,Tops,4.142857,84,0.809524,4.750000
1,Dresses,4.128205,39,0.820513,7.358974
2,Intimate,3.750000,12,0.666667,5.333333
0,Bottoms,4.400000,10,1.000000,2.000000
3,Jackets,5.000000,5,1.000000,6.400000


## 7. The `DataCleaner` pipeline

`DataCleaner` is for full workflows. It keeps the original DataFrame, lets you chain cleaning steps, and gives you a report at the end.

In [20]:
pipeline_input = reviews.copy()
pipeline_input.loc[0, "Department Name"] = " Dresses "
pipeline_input.loc[1, "Department Name"] = "Dreses"

cleaner = DataCleaner(
    pipeline_input,
    target_column="Recommended IND",
    id_columns=["Unnamed: 0", "Clothing ID"],
    categorical_columns=["Division Name", "Department Name", "Class Name"],
    numerical_columns=["Age", "Rating", "Positive Feedback Count"],
    text_columns=["Title", "Review Text"],
    verbose=False,
) #creates a DataCleaner object to clean and preprocess the dataset, specifying target, ID, categorical, numerical, and text columns

profile = cleaner.profile_data()
{
    "shape": profile["shape"],
    "duplicates": int(profile["duplicates"]),
    "quality": profile["data_quality"],
}

{'shape': (160, 11),
 'duplicates': 0,
 'quality': {'completeness': 0.94375,
  'validity': 1.0,
  'consistency': 1.0,
  'uniqueness': 1.0}}

In [21]:
(
    cleaner
    .standardize_column_names()
    .drop_columns(columns=["unnamed_0"])
    .remove_duplicates(subset=["clothing_id", "review_text"], keep="first")
    .fix_missing_values(
        strategy={
            "title": "mode",
            "review_text": "mode",
            "division_name": "mode",
            "department_name": "mode",
            "class_name": "mode",
        }
    )
    .clean_text_columns(columns=["title", "review_text"], remove_special=True, fix_typos=True)
    .remove_outliers(method="iqr", columns=["age", "positive_feedback_count"], factor=1.5)
    .convert_data_types(auto_convert=True)
) # creates a data cleaning pipeline that standardizes column names, drops unnecessary columns, removes duplicates, fixes missing values, cleans text columns, removes outliers, and converts data types automatically

cleaned = cleaner.get_cleaned_data()
cleaned.head()

,clothing_id,age,title,review_text,rating,recommended_ind,positive_feedback_count,division_name,department_name,class_name
0,767,33,huge,absolutely wonderful silky and sexy and comfortable,4,1,0,Initmates,Dresses,Intimates
1,1080,34,huge,love this dress its sooo pretty i happened to find it in a store and im glad i did bc ...,5,1,4,General,Dreses,Dresses
2,1077,60,some major design flaws,i had such high hopes for this dress and really wanted it to work for me i initially o...,3,0,0,General,Dresses,Dresses
3,1049,50,my favorite buy,i love love love this jumpsuit its fun flirty and fabulous every time i wear it i get ...,5,1,0,General Petite,Bottoms,Pants
4,847,47,flattering shirt,this shirt is very flattering to all due to the adjustable front tie it is the perfect...,5,1,6,General,Tops,Blouses


In [22]:
cleaner.compare_with_original()

,Metric,Original,Cleaned,Change,Change %
0,Rows,160.000000,133.000000,-27.000000,-16.88
1,Columns,11.000000,10.000000,-1.000000,-9.09
2,Missing Values,99.000000,0.000000,-99.000000,-100.00
3,Duplicates,0.000000,0.000000,0.000000,NaN
4,Memory (MB),0.089674,0.058948,-0.030726,-34.26


In [23]:
report = cleaner.get_report()
{
    "rows_removed": report.rows_removed,
    "columns_removed": report.columns_removed,
    "missing_values_filled": report.missing_values_filled,
    "text_corrections": report.text_corrections,
    "outliers_removed": report.outliers_removed,
    "data_types_changed": report.data_types_changed,
}

{'rows_removed': 27,
 'columns_removed': 1,
 'missing_values_filled': {'title': 42,
  'review_text': 15,
  'division_name': 10,
  'department_name': 10,
  'class_name': 10},
 'text_corrections': {'title': ['154 values cleaned'],
  'review_text': ['154 values cleaned']},
 'outliers_removed': {'age': 1, 'positive_feedback_count': 20},
 'data_types_changed': {'age': ('int64', 'int8'),
  'rating': ('int64', 'int8'),
  'positive_feedback_count': ('int64', 'int8'),
  'division_name': ('object', 'category'),
  'department_name': ('object', 'category'),
  'class_name': ('object', 'category')}}

`quick_clean()` is the one-call version. It is handy for baselines and quick experiments. For production or model-sensitive work, prefer the explicit chain so the decisions stay visible.

In [24]:
baseline = quick_clean(
    reviews.head(50),
    target_column="Recommended IND",
    id_columns=["Unnamed: 0", "Clothing ID"],
    categorical_columns=["Division Name", "Department Name", "Class Name"],
    numerical_columns=["Age", "Rating", "Positive Feedback Count"],
    text_columns=["Title", "Review Text"],
    verbose=False,
) #creates a baseline cleaned dataset using the quick_clean function, specifying target, ID, categorical, numerical, and text columns, and limiting to the first 50 rows of the reviews dataset

baseline.shape

(41, 27)

## Function map

| Function or class | Main use |
| --- | --- |
| `read_data` | Read CSV, Excel, JSON, TXT, clipboard, SAS, SPSS, Parquet, or Pickle into a DataFrame. |
| `read_images` | Load `.jpg` and `.png` files from a folder into a NumPy array. |
| `read_images_tf` | Build a TensorFlow image dataset with cache and prefetch. |
| `read_from_database` | Run a SQL query and return the result as a DataFrame. |
| `read_from_html` | Read one HTML table from a web page. |
| `drop_missing_values` | Drop rows or columns below a completeness threshold. |
| `fill_missing_values` | Fill missing values by explicit value, mean, median, mode, or forward-fill. |
| `z_remove_outliers`, `iqr_remove_outliers`, `iso_remove_outliers`, `lof_remove_outliers`, `mad_remove_outliers`, `dbscan_remove_outliers`, `percentile_remove_outliers` | Direct access to specific outlier strategies. |
| `remove_outliers` | Method dispatcher for outlier experiments. |
| `solve_data_entry_errors` | Flexible text normalization, typo repair, mapping, fuzzy matching, validation, and reporting. |
| `quick_clean_text_entries`, `strict_validate`, `smart_clean` | Convenience presets around `solve_data_entry_errors`. |
| `group_and_aggregate` | Group, aggregate, flatten multi-index columns, and reset the index. |
| `DataCleaner` | Chainable end-to-end cleaning workflow with profiling and reporting. Its structural methods include `standardize_column_names`, `rename_columns`, `drop_columns`, `drop_missing_rows`, `remove_duplicates`, `fix_missing_values`, `coerce_numeric`, and `parse_dates`. |
| `quick_clean`, `advanced_clean` | One-call wrappers around `DataCleaner`. |